# 05 — Clinic Efficiency Analysis
## Patient No-Show & Clinic Efficiency Analysis

**Objective:** quantify how much appointment capacity is actually being lost to no-shows, model realistic
recovery scenarios, and — most importantly — identify *where* that lost capacity concentrates, so management
knows exactly where to focus.

**Input:** `data/cleaned/appointments_with_risk_segments.csv` (from Notebook 04, includes `risk_segment`).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

df = pd.read_csv('../data/cleaned/appointments_with_risk_segments.csv', parse_dates=['scheduled_day','appointment_day'])
print("Shape:", df.shape)


Shape: (110521, 30)


## 1. Appointment Utilization

**Important limitation:** this dataset only contains appointments that were actually *booked* — it does not
contain a separate "total available slots offered" figure (e.g. unbooked capacity). So "utilization" here is
defined the only way the data supports: **the share of booked appointments that were actually used (attended)**,
i.e. the show rate. This is a genuine, defensible metric, but it's not the same as "% of total clinic capacity
filled" — that would require scheduling-system data this dataset doesn't have. This limitation is called out
explicitly here and again in Phase 13.


In [2]:
total_appts = len(df)
attended = (df['no_show_flag'] == 0).sum()
no_shows = (df['no_show_flag'] == 1).sum()
utilization_rate = attended / total_appts

print(f"Total booked appointments: {total_appts:,}")
print(f"Attended (utilized): {attended:,}")
print(f"No-shows (unused, booked capacity): {no_shows:,}")
print(f"Booked-capacity utilization rate: {utilization_rate:.2%}")


Total booked appointments: 110,521
Attended (utilized): 88,207
No-shows (unused, booked capacity): 22,314
Booked-capacity utilization rate: 79.81%


## 2. Lost Capacity

Every no-show is one appointment slot that was reserved, blocked off in the schedule, and then not used — and
because no-shows happen without advance warning, these slots typically cannot be backfilled from a waitlist the
way a cancellation could be.


In [3]:
lost_capacity = no_shows
print(f"Total lost appointment slots: {lost_capacity:,}")
print(f"As a share of all booked appointments: {lost_capacity/total_appts:.2%}")

# Lost capacity by month - is it concentrated or spread evenly?
lost_by_month = df[df['no_show_flag']==1].groupby(['appointment_year','appointment_month']).size()
lost_by_month = lost_by_month.reindex(pd.MultiIndex.from_tuples([(2016,'April'),(2016,'May'),(2016,'June')]))
print("\nLost slots by month:")
print(lost_by_month)


Total lost appointment slots: 22,314
As a share of all booked appointments: 20.19%

Lost slots by month:
2016  April      633
      May      16799
      June      4882
dtype: int64


## 3. Recoverable Capacity — Scenario Analysis

These are **scenario estimates**, not predictions or guaranteed outcomes — clearly labeled as such, consistent
with the project's rules. They answer: "if the clinic's no-show rate improved by X%, how many appointments would
that represent?"


In [4]:
for pct in [0.10, 0.20, 0.30]:
    recoverable = lost_capacity * pct
    print(f"Scenario: {int(pct*100)}% reduction in no-shows -> {recoverable:,.0f} appointments recovered")


Scenario: 10% reduction in no-shows -> 2,231 appointments recovered
Scenario: 20% reduction in no-shows -> 4,463 appointments recovered
Scenario: 30% reduction in no-shows -> 6,694 appointments recovered


### 3.1 A more targeted scenario: fixing the High Risk segment specifically

Rather than an across-the-board percentage cut, this scenario asks: what if outreach brought the **High Risk**
segment's no-show rate (from Notebook 04) down to just the overall average? This is a more realistic, targeted
intervention than an even reduction everywhere.


In [5]:
overall_rate = df['no_show_flag'].mean()
high_risk = df[df['risk_segment'] == 'High Risk']
high_risk_rate = high_risk['no_show_flag'].mean()
high_risk_volume = len(high_risk)
high_risk_no_shows = high_risk['no_show_flag'].sum()

# If High Risk segment's rate matched the overall average instead of its current rate:
hypothetical_no_shows_at_avg = high_risk_volume * overall_rate
recoverable_targeted = high_risk_no_shows - hypothetical_no_shows_at_avg

print(f"High Risk segment: {high_risk_volume:,} appointments, {high_risk_rate:.2%} no-show rate")
print(f"Overall average no-show rate: {overall_rate:.2%}")
print(f"Current no-shows in High Risk segment: {high_risk_no_shows:,}")
print(f"If High Risk segment matched the overall average: {hypothetical_no_shows_at_avg:,.0f} no-shows")
print(f"-> Targeted scenario recovery: {recoverable_targeted:,.0f} appointments "
      f"({recoverable_targeted/lost_capacity:.1%} of all lost capacity)")


High Risk segment: 16,374 appointments, 38.90% no-show rate
Overall average no-show rate: 20.19%
Current no-shows in High Risk segment: 6,370
If High Risk segment matched the overall average: 3,306 no-shows
-> Targeted scenario recovery: 3,064 appointments (13.7% of all lost capacity)


## 4. Operational Impact — Where Does Lost Capacity Concentrate?

This is the core of the efficiency story: not just "here's the total," but "here's where it's concentrated,"
which is what turns this into an actionable recommendation.


In [6]:
# 4.1 Concentration by risk segment (ties Phase 7 segmentation directly to capacity loss)
seg_impact = df.groupby('risk_segment').agg(
    volume=('no_show_flag','count'),
    no_shows=('no_show_flag','sum')
).reindex(['Low Risk','Medium Risk','High Risk'])
seg_impact['pct_of_volume'] = seg_impact['volume'] / total_appts
seg_impact['pct_of_lost_capacity'] = seg_impact['no_shows'] / lost_capacity
print("Lost capacity concentration by risk segment:")
print(seg_impact)


Lost capacity concentration by risk segment:
              volume  no_shows  pct_of_volume  pct_of_lost_capacity
risk_segment                                                       
Low Risk       46684      5423       0.422399              0.243031
Medium Risk    47463     10521       0.429448              0.471498
High Risk      16374      6370       0.148153              0.285471


In [7]:
# 4.2 Concentration by lead-time bucket
bucket_order = ['Same day','1-3 days','4-7 days','8-14 days','15-30 days','31+ days']
lead_impact = df.groupby('waiting_time_group').agg(
    volume=('no_show_flag','count'),
    no_shows=('no_show_flag','sum')
).reindex(bucket_order)
lead_impact['pct_of_volume'] = lead_impact['volume'] / total_appts
lead_impact['pct_of_lost_capacity'] = lead_impact['no_shows'] / lost_capacity
print("Lost capacity concentration by lead-time bucket:")
print(lead_impact)

long_lead_share = lead_impact.loc[['8-14 days','15-30 days','31+ days'], 'pct_of_lost_capacity'].sum()
long_lead_volume_share = lead_impact.loc[['8-14 days','15-30 days','31+ days'], 'pct_of_volume'].sum()
print(f"\nAppointments booked 8+ days ahead: {long_lead_volume_share:.1%} of volume, "
      f"but {long_lead_share:.1%} of all lost capacity")


Lost capacity concentration by lead-time bucket:
                    volume  no_shows  pct_of_volume  pct_of_lost_capacity
waiting_time_group                                                       
Same day             38562      1792       0.348911              0.080308
1-3 days             14675      3359       0.132780              0.150533
4-7 days             17510      4413       0.158431              0.197768
8-14 days            12025      3664       0.108803              0.164202
15-30 days           17371      5661       0.157174              0.253697
31+ days             10378      3425       0.093901              0.153491

Appointments booked 8+ days ahead: 36.0% of volume, but 57.1% of all lost capacity


In [8]:
# 4.3 High-volume + high-no-show neighbourhoods (same logic as SQL query 2.13, in Python)
neigh_stats = df.groupby('neighbourhood').agg(
    volume=('no_show_flag','count'),
    no_shows=('no_show_flag','sum')
)
neigh_stats['no_show_rate'] = neigh_stats['no_shows'] / neigh_stats['volume']
volume_p75 = neigh_stats['volume'].quantile(0.75)

priority_neighbourhoods = neigh_stats[
    (neigh_stats['volume'] >= volume_p75) & (neigh_stats['no_show_rate'] > overall_rate)
].sort_values('no_show_rate', ascending=False)

print(f"Top-quartile volume threshold: >= {volume_p75:.0f} appointments")
print(f"\n{len(priority_neighbourhoods)} neighbourhoods are BOTH high-volume AND above-average no-show rate:")
print(priority_neighbourhoods)


Top-quartile volume threshold: >= 2018 appointments

11 neighbourhoods are BOTH high-volume AND above-average no-show rate:
                   volume  no_shows  no_show_rate
neighbourhood                                    
ITARARÉ              3514       923      0.262664
JESUS DE NAZARETH    2853       696      0.243954
ILHA DO PRÍNCIPE     2266       532      0.234775
CARATOÍRA            2565       591      0.230409
ANDORINHAS           2262       521      0.230327
GURIGICA             2018       456      0.225966
ROMÃO                2214       474      0.214092
CENTRO               3334       703      0.210858
SÃO PEDRO            2448       515      0.210376
MARIA ORTIZ          5805      1219      0.209991
RESISTÊNCIA          4430       905      0.204289


## 5. Impact vs. Priority Matrix

Combining volume and no-show rate into a simple 2x2 read: which segments are both large (high volume) and
problematic (high no-show rate)? Those are the true operational priorities — a segment can be small-but-bad or
large-but-average, and neither is as urgent as large-and-bad.


In [9]:
segments_matrix = pd.DataFrame({
    'Segment': ['High Risk (score 2-3)', 'Medium Risk (score 1)', 'Low Risk (score 0)',
                'Lead time 31+ days', 'Lead time 15-30 days', 'Lead time Same day'],
    'Volume': [
        seg_impact.loc['High Risk','volume'], seg_impact.loc['Medium Risk','volume'], seg_impact.loc['Low Risk','volume'],
        lead_impact.loc['31+ days','volume'], lead_impact.loc['15-30 days','volume'], lead_impact.loc['Same day','volume']
    ],
    'No-Show Rate': [
        seg_impact.loc['High Risk','no_shows']/seg_impact.loc['High Risk','volume'],
        seg_impact.loc['Medium Risk','no_shows']/seg_impact.loc['Medium Risk','volume'],
        seg_impact.loc['Low Risk','no_shows']/seg_impact.loc['Low Risk','volume'],
        lead_impact.loc['31+ days','no_shows']/lead_impact.loc['31+ days','volume'],
        lead_impact.loc['15-30 days','no_shows']/lead_impact.loc['15-30 days','volume'],
        lead_impact.loc['Same day','no_shows']/lead_impact.loc['Same day','volume'],
    ]
})
segments_matrix['Business Impact (lost appts)'] = (segments_matrix['Volume'] * segments_matrix['No-Show Rate']).round(0).astype(int)
segments_matrix = segments_matrix.sort_values('Business Impact (lost appts)', ascending=False)

# Priority is assigned by RANK on actual business impact (lost appointments) - the most direct,
# least arbitrary way to compare a small-but-severe segment against a large-but-average one.
segments_matrix = segments_matrix.reset_index(drop=True)
n = len(segments_matrix)
segments_matrix['impact_rank'] = segments_matrix['Business Impact (lost appts)'].rank(ascending=False, method='first')

def priority(rank, n):
    if rank <= n/3:
        return 'High'
    elif rank <= 2*n/3:
        return 'Medium'
    else:
        return 'Low'

segments_matrix['Priority'] = segments_matrix['impact_rank'].apply(lambda r: priority(r, n))
segments_matrix['No-Show Rate'] = (segments_matrix['No-Show Rate']*100).round(1).astype(str) + '%'
segments_matrix = segments_matrix.drop(columns=['impact_rank'])
segments_matrix


,Segment,Volume,No-Show Rate,Business Impact (lost appts),Priority
0,Medium Risk (score 1),47463,22.2%,10521,High
1,High Risk (score 2-3),16374,38.9%,6370,High
2,Lead time 15-30 days,17371,32.6%,5661,Medium
3,Low Risk (score 0),46684,11.6%,5423,Medium
4,Lead time 31+ days,10378,33.0%,3425,Low
5,Lead time Same day,38562,4.6%,1792,Low


## 6. Summary

**Utilization:** 79.81% of booked appointments were actually used — meaning roughly 1 in 5 booked slots
(22,314 of 110,521) went unused, with no advance warning to backfill them.

**Recovery scenarios:**
- Across-the-board: 10% reduction -> 2,231 appointments recovered; 20% -> 4,463; 30% -> 6,694
- Targeted (High Risk segment only, brought down to the overall average rate): **3,064 appointments recovered**
  (13.7% of all lost capacity) — a realistic, focused intervention rather than an even cut everywhere

**Where lost capacity concentrates:**
- **Lead time is the biggest concentration effect found in the whole project:** appointments booked 8+ days
  ahead are only 36.0% of all volume, but account for **57.1% of all lost capacity**. This is the single
  clearest target for scheduling policy (e.g. earlier reconfirmation calls for long-lead-time bookings).
- By risk segment: Medium Risk contributes the most raw lost appointments (10,521 — driven by sheer volume,
  42.9% of all appointments), while High Risk contributes 6,370 despite being only 14.8% of volume — both are
  high-priority by different mechanisms (one by scale, one by rate).
- **11 neighbourhoods are both high-volume (top quartile) and above-average no-show rate** — led by ITARARÉ
  (26.3% no-show rate, 3,514 appointments) — these are the clearest location-based operational targets.
- Notably, even the **Low Risk segment still contributes 5,423 lost appointments** in raw terms, purely because
  of its size (42.2% of all volume) — a reminder that "low risk" doesn't mean "zero impact," and some baseline
  no-show reduction effort (e.g. universal reminders) still has value even for the safest-looking segment.

**Framing:** utilization here means "share of booked appointments attended," since this dataset has no record of
unbooked/available capacity — a limitation worth stating plainly rather than implying a fuller picture than the
data supports.